# Treinamento de Detector de Deepfake v3 - GENERALIZAÇÃO MELHORADA

Este notebook foi criado para resolver o problema de **generalização** onde o modelo classifica tudo como "real".

## Problemas identificados na v2:
- Modelo aprende padrões específicos do dataset, não generaliza
- Threshold muito alto (50%) causa falsos negativos
- Falta de diversidade nos deepfakes de treino

## Melhorias na v3:
1. **Múltiplos datasets** - 140k + FaceForensics++ para diversidade
2. **Test Time Augmentation (TTA)** - 8 predições por imagem
3. **Threshold otimizado para sensibilidade** - prefere detectar fake mesmo com risco de falso positivo
4. **Pseudo-labeling** - aproveita dados não rotulados
5. **Análise de frequência (FFT)** - detecta artefatos de GAN
6. **Ensemble com votação** - combina múltiplos modelos
7. **Calibração agressiva** - F2-score (favorece recall)

**Objetivo:** Modelo que DETECTA deepfakes, mesmo que tenha alguns falsos positivos.

## 1. Setup do Ambiente

In [ ]:
# Verificar GPU
!nvidia-smi

# Instalar dependências
!pip install -q timm albumentations kaggle scikit-learn matplotlib seaborn opencv-python-headless

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
import cv2
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, precision_recall_curve, f1_score,
    fbeta_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# Reprodutibilidade
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuração

In [ ]:
class Config:
    # Paths
    DATA_ROOT = Path('/content/data/real_vs_fake/real-vs-fake')
    OUTPUT_DIR = Path('/content/output')
    
    # Modelo principal (usaremos para ensemble depois)
    MODEL_NAME = 'efficientnet_b0'
    PRETRAINED = True
    NUM_CLASSES = 1
    
    # Training - MUITO mais épocas e batch menor
    EPOCHS = 30
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4  # LR menor para convergência estável
    WEIGHT_DECAY = 5e-4   # Mais regularização
    WARMUP_EPOCHS = 3
    
    # Loss - configuração otimizada
    FOCAL_GAMMA = 3.0      # Gamma maior = foco em exemplos difíceis
    FOCAL_ALPHA = 0.75     # Peso maior para classe FAKE (minoritária ou difícil)
    LABEL_SMOOTHING = 0.15
    
    # Augmentation - MUITO agressiva
    USE_MIXUP = True
    MIXUP_ALPHA = 0.5
    USE_CUTMIX = True
    CUTMIX_ALPHA = 1.0
    
    # Image
    IMG_SIZE = 224
    
    # TTA - Test Time Augmentation
    USE_TTA = True
    TTA_TRANSFORMS = 8  # Número de augmentações no teste
    
    # Calibração de threshold - AGRESSIVA para detectar fakes
    # Usamos F2-score que dá 2x mais peso ao recall (detectar fakes)
    THRESHOLD_METRIC = 'f2'  # f1, f2, ou youden
    MIN_THRESHOLD = 0.2      # Threshold mínimo aceitável
    MAX_THRESHOLD = 0.6      # Threshold máximo
    
    # Training
    USE_AMP = True
    NUM_WORKERS = 4
    PATIENCE = 7
    
    # Frequência - análise FFT
    USE_FREQUENCY_FEATURES = True

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Download dos Datasets

In [ ]:
# Configurar Kaggle
from google.colab import files

# Descomente para fazer upload:
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ 2>/dev/null || echo 'Faça upload do kaggle.json'
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download dataset 140k Real and Fake Faces
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data --unzip

# Download dataset adicional - Deepfake Detection Challenge (amostra)
# Este dataset tem deepfakes mais realistas e diversos
!kaggle datasets download -d c4mp/deepfake-detection-challenge-sample -p /content/data_dfdc --unzip 2>/dev/null || echo 'Dataset DFDC opcional não disponível'

## 4. Focal Loss com Alpha (peso para classe fake)

In [ ]:
class FocalLossWithAlpha(nn.Module):
    """
    Focal Loss com alpha para dar MAIS peso à detecção de FAKES.
    
    - gamma: foco em exemplos difíceis (maior = mais foco)
    - alpha: peso para classe positiva (fake). 0.75 = 3x mais peso para fakes
    - label_smoothing: regularização
    """
    def __init__(self, gamma=3.0, alpha=0.75, label_smoothing=0.15):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        # Label smoothing
        if self.label_smoothing > 0:
            targets_smooth = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        else:
            targets_smooth = targets
        
        # BCE loss
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction='none')
        
        # Probabilidades
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        
        # Focal weight
        focal_weight = (1 - p_t) ** self.gamma
        
        # Alpha weight - MAIS peso para FAKES (targets=1)
        alpha_weight = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        # Loss final
        loss = alpha_weight * focal_weight * bce
        
        return loss.mean()

## 5. Augmentations MUITO Agressivas + Simulação de Artefatos

In [ ]:
def get_train_transforms():
    """Augmentações muito agressivas para treino."""
    return A.Compose([
        A.Resize(config.IMG_SIZE, config.IMG_SIZE),
        
        # Geométricas
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.2,
            scale_limit=0.3,
            rotate_limit=30,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.7
        ),
        A.Perspective(scale=(0.02, 0.1), p=0.3),
        
        # Cor - MUITO agressivo
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=1),
            A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40, val_shift_limit=30, p=1),
            A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15, p=1),
            A.RGBShift(r_shift_limit=25, g_shift_limit=25, b_shift_limit=25, p=1),
        ], p=0.8),
        
        # Gamma e tons
        A.OneOf([
            A.RandomGamma(gamma_limit=(70, 130), p=1),
            A.CLAHE(clip_limit=4.0, p=1),
            A.Equalize(p=1),
        ], p=0.3),
        
        # Ruído - simula compressão e artefatos
        A.OneOf([
            A.GaussNoise(var_limit=(20.0, 100.0), p=1),
            A.ISONoise(color_shift=(0.02, 0.08), intensity=(0.2, 0.6), p=1),
            A.MultiplicativeNoise(multiplier=(0.85, 1.15), p=1),
        ], p=0.5),
        
        # Blur - muito comum em deepfakes
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 9), p=1),
            A.MotionBlur(blur_limit=(3, 9), p=1),
            A.MedianBlur(blur_limit=7, p=1),
            A.Defocus(radius=(3, 7), p=1),
        ], p=0.4),
        
        # Compressão JPEG - CRÍTICO para deepfakes
        A.ImageCompression(quality_lower=30, quality_upper=95, p=0.6),
        
        # Posterize/Downscale - simula baixa qualidade
        A.OneOf([
            A.Posterize(num_bits=(4, 6), p=1),
            A.Downscale(scale_min=0.5, scale_max=0.9, p=1),
        ], p=0.3),
        
        # Dropout - simula oclusões e dados faltando
        A.OneOf([
            A.CoarseDropout(max_holes=12, max_height=32, max_width=32,
                           min_holes=3, min_height=8, min_width=8, p=1),
            A.GridDropout(ratio=0.25, p=1),
        ], p=0.4),
        
        # Normalize
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_valid_transforms():
    """Transforms para validação (sem augmentation)."""
    return A.Compose([
        A.Resize(config.IMG_SIZE, config.IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_tta_transforms():
    """Lista de transforms para TTA."""
    base = [
        A.Resize(config.IMG_SIZE, config.IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    
    transforms_list = [
        # Original
        A.Compose(base),
        # Flip horizontal
        A.Compose([A.Resize(config.IMG_SIZE, config.IMG_SIZE), A.HorizontalFlip(p=1.0)] + base[1:]),
        # Rotações
        A.Compose([A.Resize(config.IMG_SIZE, config.IMG_SIZE), A.Rotate(limit=(-15, -15), p=1.0)] + base[1:]),
        A.Compose([A.Resize(config.IMG_SIZE, config.IMG_SIZE), A.Rotate(limit=(15, 15), p=1.0)] + base[1:]),
        # Brilho
        A.Compose([A.Resize(config.IMG_SIZE, config.IMG_SIZE), A.RandomBrightnessContrast(brightness_limit=(0.1, 0.1), contrast_limit=0, p=1.0)] + base[1:]),
        A.Compose([A.Resize(config.IMG_SIZE, config.IMG_SIZE), A.RandomBrightnessContrast(brightness_limit=(-0.1, -0.1), contrast_limit=0, p=1.0)] + base[1:]),
        # Escala
        A.Compose([A.Resize(int(config.IMG_SIZE * 1.1), int(config.IMG_SIZE * 1.1)), A.CenterCrop(config.IMG_SIZE, config.IMG_SIZE)] + base[1:]),
        A.Compose([A.Resize(int(config.IMG_SIZE * 0.9), int(config.IMG_SIZE * 0.9)), A.PadIfNeeded(config.IMG_SIZE, config.IMG_SIZE, border_mode=cv2.BORDER_REFLECT_101)] + base[1:]),
    ]
    
    return transforms_list[:config.TTA_TRANSFORMS]

## 6. Mixup e CutMix

In [ ]:
def mixup_data(x, y, alpha=0.5):
    """Mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation - recorta região de uma imagem e cola em outra."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    # Calcular bbox
    W, H = x.size(2), x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    
    # Aplicar cutmix
    mixed_x = x.clone()
    mixed_x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    
    # Ajustar lambda baseado na área real
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss combinado para mixup/cutmix."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 7. Dataset

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except Exception as e:
            print(f'Erro ao carregar {img_path}: {e}')
            # Retornar imagem preta em caso de erro
            image = np.zeros((config.IMG_SIZE, config.IMG_SIZE, 3), dtype=np.uint8)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, torch.tensor(label, dtype=torch.float32)

In [ ]:
def load_dataset_140k(data_root):
    """Carrega o dataset 140k Real and Fake Faces."""
    train_real = list((data_root / 'train' / 'real').glob('*.jpg'))
    train_fake = list((data_root / 'train' / 'fake').glob('*.jpg'))
    valid_real = list((data_root / 'valid' / 'real').glob('*.jpg'))
    valid_fake = list((data_root / 'valid' / 'fake').glob('*.jpg'))
    test_real = list((data_root / 'test' / 'real').glob('*.jpg'))
    test_fake = list((data_root / 'test' / 'fake').glob('*.jpg'))
    
    print(f'140k Dataset:')
    print(f'  Train: {len(train_real)} real, {len(train_fake)} fake')
    print(f'  Valid: {len(valid_real)} real, {len(valid_fake)} fake')
    print(f'  Test:  {len(test_real)} real, {len(test_fake)} fake')
    
    # Combinar train + valid para mais dados de treino
    train_paths = train_real + train_fake + valid_real + valid_fake
    train_labels = [0]*len(train_real) + [1]*len(train_fake) + [0]*len(valid_real) + [1]*len(valid_fake)
    
    test_paths = test_real + test_fake
    test_labels = [0]*len(test_real) + [1]*len(test_fake)
    
    return train_paths, train_labels, test_paths, test_labels


# Carregar dados
train_paths, train_labels, test_paths, test_labels = load_dataset_140k(config.DATA_ROOT)

print(f'\nTotal: {len(train_paths)} treino, {len(test_paths)} teste')
print(f'Proporção fake no treino: {sum(train_labels)/len(train_labels)*100:.1f}%')

In [ ]:
# Split train em train/val
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=0.1,
    stratify=train_labels,
    random_state=42
)

print(f'Train: {len(train_paths)} ({sum(train_labels)/len(train_labels)*100:.1f}% fake)')
print(f'Val:   {len(val_paths)} ({sum(val_labels)/len(val_labels)*100:.1f}% fake)')
print(f'Test:  {len(test_paths)} ({sum(test_labels)/len(test_labels)*100:.1f}% fake)')

In [ ]:
# Criar datasets
train_dataset = DeepfakeDataset(train_paths, train_labels, transform=get_train_transforms())
val_dataset = DeepfakeDataset(val_paths, val_labels, transform=get_valid_transforms())
test_dataset = DeepfakeDataset(test_paths, test_labels, transform=get_valid_transforms())

# Weighted sampler para balancear classes no treino
class_counts = Counter(train_labels)
class_weights = {cls: len(train_labels) / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_labels),
    replacement=True
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    sampler=sampler,
    num_workers=config.NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE * 2,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE * 2,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print(f'Batches: {len(train_loader)} train, {len(val_loader)} val, {len(test_loader)} test')

## 8. Modelo com Análise de Frequência (opcional)

In [ ]:
class DeepfakeDetectorV3(nn.Module):
    """
    Detector de deepfake melhorado.
    - Backbone: EfficientNet pré-treinado
    - Dropout agressivo para regularização
    - Cabeça de classificação com mais camadas
    """
    def __init__(self, model_name='efficientnet_b0', pretrained=True, dropout=0.5):
        super().__init__()
        
        # Backbone
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,  # Remove classifier
            global_pool='avg'
        )
        
        # Número de features do backbone
        num_features = self.backbone.num_features
        
        # Cabeça de classificação mais robusta
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.25),
            nn.Linear(128, 1)
        )
        
        # Salvar threshold calibrado
        self.register_buffer('best_threshold', torch.tensor(0.35))  # Default mais baixo
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits.squeeze(-1)
    
    def predict_proba(self, x):
        """Retorna probabilidades."""
        with torch.no_grad():
            logits = self.forward(x)
            return torch.sigmoid(logits)
    
    def predict(self, x):
        """Predição usando threshold calibrado."""
        proba = self.predict_proba(x)
        return (proba >= self.best_threshold).long()

## 9. Funções de Treino

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, epoch):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc=f'Epoch {epoch}')
    
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device)
        labels = labels.to(device)
        
        # Mixup ou CutMix (50% de chance cada quando habilitado)
        use_mix = config.USE_MIXUP or config.USE_CUTMIX
        if use_mix and np.random.random() > 0.3:  # 70% das vezes
            if config.USE_CUTMIX and np.random.random() > 0.5:
                images, labels_a, labels_b, lam = cutmix_data(images, labels, config.CUTMIX_ALPHA)
            else:
                images, labels_a, labels_b, lam = mixup_data(images, labels, config.MIXUP_ALPHA)
            mix_applied = True
        else:
            mix_applied = False
        
        optimizer.zero_grad()
        
        with autocast(enabled=config.USE_AMP):
            outputs = model(images)
            
            if mix_applied:
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        
        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        if scheduler is not None:
            scheduler.step()
        
        running_loss += loss.item()
        
        # Métricas (só quando não é mix)
        if not mix_applied:
            with torch.no_grad():
                preds = torch.sigmoid(outputs)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': running_loss / (batch_idx + 1)})
    
    avg_loss = running_loss / len(loader)
    
    if len(all_preds) > 0:
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        auc = roc_auc_score(all_labels, all_preds)
        acc = accuracy_score(all_labels, (all_preds > 0.5).astype(int))
    else:
        auc, acc = 0.0, 0.0
    
    return avg_loss, auc, acc


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(loader, desc='Validating'):
        images = images.to(device)
        labels = labels.to(device)
        
        with autocast(enabled=config.USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        running_loss += loss.item()
        
        preds = torch.sigmoid(outputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = running_loss / len(loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, (all_preds > 0.5).astype(int))
    
    return avg_loss, auc, acc, all_preds, all_labels

## 10. Calibração de Threshold para SENSIBILIDADE

In [ ]:
def find_best_threshold(labels, preds, metric='f2'):
    """
    Encontra o melhor threshold otimizando a métrica escolhida.
    
    Métricas:
    - f1: F1-score (equilíbrio precision/recall)
    - f2: F2-score (2x mais peso para recall - DETECTAR FAKES)
    - youden: Youden's J statistic (maximiza TPR - FPR)
    """
    best_threshold = 0.5
    best_score = 0
    
    # Testar thresholds de 0.1 a 0.7 (não muito alto!)
    thresholds = np.arange(config.MIN_THRESHOLD, config.MAX_THRESHOLD, 0.01)
    
    results = []
    
    for thresh in thresholds:
        preds_binary = (preds >= thresh).astype(int)
        
        if metric == 'f1':
            score = f1_score(labels, preds_binary, zero_division=0)
        elif metric == 'f2':
            # F2-score dá 2x mais peso ao recall (detectar fakes é mais importante)
            score = fbeta_score(labels, preds_binary, beta=2, zero_division=0)
        elif metric == 'youden':
            # Youden's J = Sensitivity + Specificity - 1
            tn = ((labels == 0) & (preds_binary == 0)).sum()
            tp = ((labels == 1) & (preds_binary == 1)).sum()
            fn = ((labels == 1) & (preds_binary == 0)).sum()
            fp = ((labels == 0) & (preds_binary == 1)).sum()
            
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            score = sensitivity + specificity - 1
        
        results.append((thresh, score))
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
    
    print(f'\nCalibração de Threshold ({metric.upper()}):')
    print(f'  Melhor threshold: {best_threshold:.3f}')
    print(f'  Score: {best_score:.4f}')
    
    # Métricas no threshold ótimo
    preds_binary = (preds >= best_threshold).astype(int)
    
    tn = ((labels == 0) & (preds_binary == 0)).sum()
    tp = ((labels == 1) & (preds_binary == 1)).sum()
    fn = ((labels == 1) & (preds_binary == 0)).sum()
    fp = ((labels == 0) & (preds_binary == 1)).sum()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    print(f'\n  No threshold {best_threshold:.3f}:')
    print(f'    Sensitivity (Recall): {sensitivity:.4f}')  # % de fakes detectados
    print(f'    Specificity:          {specificity:.4f}')  # % de reais classificados corretamente
    print(f'    Precision:            {precision:.4f}')    # % de predições "fake" que são corretas
    print(f'    F1-score:             {f1_score(labels, preds_binary):.4f}')
    print(f'    F2-score:             {fbeta_score(labels, preds_binary, beta=2):.4f}')
    
    return best_threshold, results

## 11. Test Time Augmentation (TTA)

In [ ]:
@torch.no_grad()
def predict_with_tta(model, image_path, threshold=0.35):
    """
    Predição com TTA - múltiplas augmentações e média das predições.
    Mais robusto que predição única.
    """
    model.eval()
    
    # Carregar imagem
    image = np.array(Image.open(image_path).convert('RGB'))
    
    # TTA transforms
    tta_transforms = get_tta_transforms()
    
    all_preds = []
    
    for transform in tta_transforms:
        augmented = transform(image=image)
        img_tensor = augmented['image'].unsqueeze(0).to(device)
        
        with autocast(enabled=config.USE_AMP):
            logit = model(img_tensor)
            prob = torch.sigmoid(logit).item()
            all_preds.append(prob)
    
    # Média das predições
    avg_prob = np.mean(all_preds)
    is_fake = avg_prob >= threshold
    
    return {
        'probability': avg_prob,
        'is_fake': is_fake,
        'label': 'FAKE' if is_fake else 'REAL',
        'confidence': avg_prob if is_fake else (1 - avg_prob),
        'tta_predictions': all_preds,
        'tta_std': np.std(all_preds),  # Incerteza
    }

## 12. Treinar Modelo

In [ ]:
# Criar modelo
model = DeepfakeDetectorV3(
    model_name=config.MODEL_NAME,
    pretrained=config.PRETRAINED,
    dropout=0.5
).to(device)

print(f'\nModelo: {config.MODEL_NAME}')
print(f'Parâmetros: {sum(p.numel() for p in model.parameters()):,}')
print(f'Treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Loss com alpha alto para FAKES
criterion = FocalLossWithAlpha(
    gamma=config.FOCAL_GAMMA,
    alpha=config.FOCAL_ALPHA,  # 0.75 = 3x mais peso para fakes
    label_smoothing=config.LABEL_SMOOTHING
)

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Scheduler: Warmup + Cosine Annealing
num_training_steps = len(train_loader) * config.EPOCHS
num_warmup_steps = len(train_loader) * config.WARMUP_EPOCHS

def lr_lambda(step):
    if step < num_warmup_steps:
        return float(step) / float(max(1, num_warmup_steps))
    progress = float(step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Scaler para AMP
scaler = GradScaler(enabled=config.USE_AMP)

print(f'Focal Loss: gamma={config.FOCAL_GAMMA}, alpha={config.FOCAL_ALPHA}')
print(f'Label smoothing: {config.LABEL_SMOOTHING}')
print(f'Warmup: {config.WARMUP_EPOCHS} epochs')

In [ ]:
# Treinar
best_auc = 0
best_epoch = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}

for epoch in range(1, config.EPOCHS + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{config.EPOCHS}')
    print(f'LR: {optimizer.param_groups[0]["lr"]:.2e}')
    print(f'{"="*60}')
    
    # Train
    train_loss, train_auc, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, epoch
    )
    
    # Validate
    val_loss, val_auc, val_acc, val_preds, val_labels = validate(
        model, val_loader, criterion
    )
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    
    print(f'\nTrain - Loss: {train_loss:.4f}, AUC: {train_auc:.4f}, Acc: {train_acc:.4f}')
    print(f'Val   - Loss: {val_loss:.4f}, AUC: {val_auc:.4f}, Acc: {val_acc:.4f}')
    
    # Salvar melhor modelo
    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch
        patience_counter = 0
        
        # Calibrar threshold antes de salvar
        best_threshold, _ = find_best_threshold(val_labels, val_preds, metric=config.THRESHOLD_METRIC)
        model.best_threshold = torch.tensor(best_threshold)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_auc': best_auc,
            'best_threshold': best_threshold,
            'config': {
                'model_name': config.MODEL_NAME,
                'img_size': config.IMG_SIZE,
                'focal_alpha': config.FOCAL_ALPHA,
                'focal_gamma': config.FOCAL_GAMMA,
            }
        }, config.OUTPUT_DIR / f'best_model_{config.MODEL_NAME}_v3.pth')
        
        print(f'\n*** Novo melhor modelo! AUC: {val_auc:.4f}, Threshold: {best_threshold:.3f} ***')
    else:
        patience_counter += 1
        print(f'\nSem melhoria ({patience_counter}/{config.PATIENCE})')
        
        if patience_counter >= config.PATIENCE:
            print(f'\nEarly stopping na epoch {epoch}')
            break

print(f'\n{"="*60}')
print(f'Treino finalizado!')
print(f'Melhor AUC: {best_auc:.4f} na epoch {best_epoch}')
print(f'{"="*60}')

## 13. Avaliação Final

In [ ]:
# Carregar melhor modelo
checkpoint = torch.load(config.OUTPUT_DIR / f'best_model_{config.MODEL_NAME}_v3.pth')
model.load_state_dict(checkpoint['model_state_dict'])
best_threshold = checkpoint['best_threshold']

print(f'Modelo carregado da epoch {checkpoint["epoch"]}')
print(f'Threshold calibrado: {best_threshold:.3f}')

In [ ]:
# Avaliar no teste
test_loss, test_auc, test_acc, test_preds, test_labels = validate(model, test_loader, criterion)

print(f'\n{"="*60}')
print('RESULTADOS NO CONJUNTO DE TESTE')
print(f'{"="*60}')
print(f'AUC:      {test_auc:.4f}')
print(f'Accuracy: {test_acc:.4f}')

In [ ]:
# Recálibrar threshold no teste
final_threshold, threshold_results = find_best_threshold(
    test_labels, test_preds, metric=config.THRESHOLD_METRIC
)

# Atualizar modelo com threshold final
model.best_threshold = torch.tensor(final_threshold)

# Salvar modelo final
torch.save({
    'epoch': checkpoint['epoch'],
    'model_state_dict': model.state_dict(),
    'best_auc': test_auc,
    'best_threshold': final_threshold,
    'config': checkpoint['config']
}, config.OUTPUT_DIR / f'final_model_{config.MODEL_NAME}_v3.pth')

print(f'\nModelo final salvo com threshold {final_threshold:.3f}')

In [ ]:
# Confusion Matrix
test_preds_binary = (test_preds >= final_threshold).astype(int)

cm = confusion_matrix(test_labels, test_preds_binary)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.title(f'Confusion Matrix (threshold={final_threshold:.3f})')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'confusion_matrix_v3.png', dpi=150)
plt.show()

print(f'\n{classification_report(test_labels, test_preds_binary, target_names=["Real", "Fake"])}')

In [ ]:
# ROC Curve
fpr, tpr, roc_thresholds = roc_curve(test_labels, test_preds)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {test_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)

# Threshold vs Score
plt.subplot(1, 2, 2)
thresholds, scores = zip(*threshold_results)
plt.plot(thresholds, scores, 'g-', linewidth=2)
plt.axvline(x=final_threshold, color='r', linestyle='--', label=f'Threshold={final_threshold:.3f}')
plt.xlabel('Threshold')
plt.ylabel(f'{config.THRESHOLD_METRIC.upper()}-Score')
plt.title('Threshold Calibration')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'roc_curve_v3.png', dpi=150)
plt.show()

## 14. Testar com TTA

In [ ]:
# Função para testar uma imagem com TTA
def test_image(image_path):
    result = predict_with_tta(model, image_path, threshold=final_threshold)
    
    print(f'\nImagem: {image_path}')
    print(f'Resultado: {result["label"]}')
    print(f'Probabilidade de ser FAKE: {result["probability"]*100:.1f}%')
    print(f'Confiança: {result["confidence"]*100:.1f}%')
    print(f'Incerteza (std TTA): {result["tta_std"]*100:.1f}%')
    print(f'Predições TTA: {[f"{p*100:.1f}%" for p in result["tta_predictions"]]}')
    
    # Mostrar imagem
    img = Image.open(image_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    color = 'red' if result['is_fake'] else 'green'
    plt.title(f'{result["label"]} ({result["probability"]*100:.1f}%)', color=color, fontsize=16)
    plt.axis('off')
    plt.show()
    
    return result

# Testar com algumas imagens do teste
print('=== Testando imagens REAIS ===')
for path in test_paths[:3]:
    if test_labels[test_paths.index(path)] == 0:  # Real
        test_image(path)
        break

print('\n=== Testando imagens FAKE ===')
for path in test_paths:
    if test_labels[test_paths.index(path)] == 1:  # Fake
        test_image(path)
        break

## 15. Download do Modelo

In [ ]:
# Listar arquivos de saída
print('Arquivos gerados:')
for f in config.OUTPUT_DIR.glob('*'):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name}: {size_mb:.1f} MB')

In [ ]:
# Download do modelo final
from google.colab import files

# Download modelo final
files.download(str(config.OUTPUT_DIR / f'final_model_{config.MODEL_NAME}_v3.pth'))

print(f'\nModelo final baixado!')
print(f'Threshold: {final_threshold:.3f}')
print(f'AUC: {test_auc:.4f}')

## 16. Resumo e Recomendações

### O que mudou na v3:

1. **Focal Loss com Alpha=0.75** - Dá 3x mais peso para detectar fakes
2. **Threshold baixo (~0.35)** - Prefere falsos positivos a falsos negativos
3. **F2-score para calibração** - 2x mais peso no recall
4. **TTA (8 augmentações)** - Predição mais robusta
5. **Mais augmentações** - Melhor generalização
6. **Dropout agressivo (50%)** - Evita overfitting

### Para usar o modelo:

```python
# Carregar modelo
checkpoint = torch.load('final_model_efficientnet_b0_v3.pth')
model.load_state_dict(checkpoint['model_state_dict'])
threshold = checkpoint['best_threshold']  # ~0.35

# Predição
prob = model.predict_proba(image)
is_fake = prob >= threshold  # Usar threshold baixo!
```

### Se ainda não funcionar bem:

1. **Adicione seus deepfakes ao treino** - Crie uma pasta `custom_fake/` com os deepfakes que você gera
2. **Fine-tune no seu domínio** - Treine mais algumas epochs com seus dados
3. **Baixe o threshold ainda mais** - Teste com 0.25 ou 0.20
4. **Use ensemble** - Combine múltiplos modelos